# ASA DataFest 2026 — Stormont Vail Health
## Exploratory Data Analysis Notebook

This notebook walks through the seven dataset tables systematically:
1. Load & overview
2. Null / missingness audit
3. Encounters (core fact table)
4. Patients (demographics)
5. Diagnosis (ICD-10)
6. Providers
7. Departments
8. Social Determinants of Health (SDOH)
9. Join validation
10. Patient journey setup
11. Demographics × encounters

> **Data privacy reminder:** Do not upload these files to public services, GitHub, or external APIs.

In [ ]:
import warnings
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

---
## 1. Load All Tables

In [ ]:
DATA = '2026-ASA-DataFest-Data-Files/'

print('Loading tables...', flush=True)
enc   = pd.read_csv(DATA + 'encounters.csv',          low_memory=False, parse_dates=['Date'])
print(f'  encounters:          {len(enc):>10,} rows')

pat   = pd.read_csv(DATA + 'patients.csv',            low_memory=False)
print(f'  patients:            {len(pat):>10,} rows')

diag  = pd.read_csv(DATA + 'diagnosis.csv',           low_memory=False)
print(f'  diagnosis:           {len(diag):>10,} rows')

prov  = pd.read_csv(DATA + 'providers.csv',           low_memory=False)
print(f'  providers:           {len(prov):>10,} rows')

dept  = pd.read_csv(DATA + 'departments.csv',         low_memory=False)
print(f'  departments:         {len(dept):>10,} rows')

sdoh  = pd.read_csv(DATA + 'social_determinants.csv', low_memory=False)
print(f'  social_determinants: {len(sdoh):>10,} rows')

tiger = pd.read_csv(DATA + 'tigercensuscodes.csv',    low_memory=False)
print(f'  tigercensus:         {len(tiger):>10,} rows')

In [ ]:
# Memory usage summary
tables = dict(encounters=enc, patients=pat, diagnosis=diag,
              providers=prov, departments=dept,
              social_determinants=sdoh, tigercensus=tiger)

summary = pd.DataFrame([
    {'table': k, 'rows': len(v), 'cols': v.shape[1],
     'mem_mb': round(v.memory_usage(deep=True).sum() / 1e6, 1)}
    for k, v in tables.items()
]).set_index('table')

summary

---
## 2. Null / Missingness Audit

**DataFest convention:**
- `*Unspecified` — field was asked but patient didn't answer
- `*Unknown` — field left blank (unknown reason)
- `*Not Applicable` — field doesn't apply to this record
- `-1` in diagnosis key — no diagnosis given (valid, filter before joining)

In [ ]:
def null_audit(df):
    rows = []
    for col in df.columns:
        n_null   = df[col].isna().sum()
        n_unspec = df[col].astype(str).str.startswith('*').sum() if df[col].dtype == object else 0
        rows.append({
            'column':              col,
            'dtype':               str(df[col].dtype),
            'n_null':              n_null,
            'n_unspecified':       n_unspec,
            'pct_missing_or_unspec': round((n_null + n_unspec) / len(df) * 100, 1),
            'n_unique':            df[col].nunique(),
        })
    return pd.DataFrame(rows).set_index('column')

for name, df in tables.items():
    audit = null_audit(df)
    problem = audit[audit['pct_missing_or_unspec'] > 0][['pct_missing_or_unspec','n_null','n_unspecified']]
    if not problem.empty:
        print(f'\n=== {name} ===')
        display(problem.style.background_gradient(subset=['pct_missing_or_unspec'], cmap='Reds'))

---
## 3. Encounters — Core Fact Table

In [ ]:
enc['Year']      = enc['Date'].dt.year
enc['Month']     = enc['Date'].dt.month
enc['YearMonth'] = enc['Date'].dt.to_period('M')

print('Column overview:')
enc.dtypes.to_frame('dtype')

In [ ]:
# Encounter volume by year
vol_year = enc['Year'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
vol_year.plot(kind='bar', ax=ax, color=sns.color_palette('muted')[0])
ax.set_title('Encounters by Year')
ax.set_xlabel('Year'); ax.set_ylabel('Encounters')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

print(vol_year)

In [ ]:
# Monthly trend
vol_month = enc.groupby('YearMonth').size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(vol_month['YearMonth'].astype(str), vol_month['count'], marker='o', markersize=3)
ax.set_title('Monthly Encounter Volume (2022–2025)')
ax.set_xlabel('Month'); ax.set_ylabel('Encounters')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Encounter type distribution
type_counts = enc['Type'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
type_counts.plot(kind='bar', ax=ax, color=sns.color_palette('muted'))
ax.set_title('Encounter Type Distribution')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# Boolean encounter flags
bool_flags = [c for c in enc.columns if c.startswith('Is')]
flag_counts = pd.Series({f: int(enc[f].sum()) for f in bool_flags}).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
flag_counts.plot(kind='bar', ax=ax, color=sns.color_palette('Set2'))
ax.set_title('Encounter Volume by Boolean Flag')
ax.set_ylabel('True count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# Hospital length of stay
hosp = enc[enc['IsHospitalAdmission'] == True].copy()
hosp['AdmissionInstant'] = pd.to_datetime(hosp['AdmissionInstant'], errors='coerce')
hosp['DischargeInstant'] = pd.to_datetime(hosp['DischargeInstant'],  errors='coerce')
hosp['LOS_days'] = (hosp['DischargeInstant'] - hosp['AdmissionInstant']).dt.total_seconds() / 86400
hosp_valid = hosp[(hosp['LOS_days'] > 0) & (hosp['LOS_days'] < 365)]

print(f'Hospital admissions with valid LOS: {len(hosp_valid):,}')
print(hosp_valid['LOS_days'].describe().round(2))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(hosp_valid['LOS_days'].clip(upper=30), bins=60,
        color=sns.color_palette('muted')[2], edgecolor='white')
ax.set_title('Hospital Length of Stay (capped at 30 days)')
ax.set_xlabel('Days'); ax.set_ylabel('Encounters')
plt.tight_layout(); plt.show()

---
## 4. Patients — Demographics

In [ ]:
pat.head(3)

In [ ]:
# Age distribution (birth year bins)
# NOTE: PatientBirthYearBin is the UPPER bound of a 5-year range.
# e.g. bin 1970 = born 1966–1970. Approximate age = reference_year - PatientBirthYearBin
birth_counts = pat['PatientBirthYearBin'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
birth_counts.plot(kind='bar', ax=ax, color=sns.color_palette('muted')[1])
ax.set_title('Patient Age Distribution (Birth Year Bin = upper bound, e.g. 1970 = born 1966–1970)')
ax.set_xlabel('Birth Year Bin (upper bound)'); ax.set_ylabel('Patients')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Race distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ['FirstRace', 'OmbEthnicity']):
    vc = pat[col].value_counts().head(10)
    ax.barh(range(len(vc)), vc.values)
    ax.set_yticks(range(len(vc)))
    ax.set_yticklabels(vc.index, fontsize=8)
    ax.set_title(col)
    ax.invert_yaxis()

plt.suptitle('Patient Race / Ethnicity')
plt.tight_layout(); plt.show()

In [ ]:
# Marital status, smoking, vital status, MyChart
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, col in zip(axes, ['MaritalStatus', 'SmokingStatus', 'VitalStatus', 'MyChartStatus']):
    vc = pat[col].value_counts().head(8)
    ax.barh(range(len(vc)), vc.values)
    ax.set_yticks(range(len(vc)))
    ax.set_yticklabels([textwrap.fill(str(v), 22) for v in vc.index], fontsize=7)
    ax.set_title(col, fontsize=9)
    ax.invert_yaxis()

plt.suptitle('Patient Demographic Variables')
plt.tight_layout(); plt.show()

In [ ]:
# Geographic coverage
geo_known  = pat[~pat['CensusBlockGroupFipsCode'].astype(str).str.startswith('*')]
geo_unspec = pat[pat['CensusBlockGroupFipsCode'].astype(str).str.startswith('*')]
pct_known  = len(geo_known) / len(pat) * 100

print(f'Patients with known census block:    {len(geo_known):>7,}  ({pct_known:.1f}%)')
print(f'Patients with suppressed/unspecified:{len(geo_unspec):>7,}  ({100-pct_known:.1f}%)')
print('\nNote: rural blocks with <11 patients are suppressed for privacy.')

---
## 5. Diagnosis — ICD-10-CM Codes

In [ ]:
diag.head(3)

In [ ]:
print(f'Unique ICD-10 codes:  {diag["DiagnosisValue"].nunique():,}')
print(f'Unique ICD-10 groups: {diag["GroupCode"].nunique():,}')

In [ ]:
# Top diagnosis groups
group_counts = diag['GroupName'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 7))
group_counts.plot(kind='barh', ax=ax, color=sns.color_palette('muted')[3])
ax.set_title('Top 20 Diagnosis Groups (ICD-10 GroupName)')
ax.invert_yaxis()
ax.set_xlabel('Count')
ax.yaxis.set_tick_params(labelsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Top specific diagnoses (excluding IMO reserved codes)
diag_clean = diag[~diag['DiagnosisValue'].isin(['IMO0001', 'IMO0002'])]
diag_clean['DiagnosisName'].value_counts().head(15)

---
## 6. Providers

In [ ]:
prov.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col in zip(axes, ['Type', 'PrimarySpecialty']):
    vc = prov[col].value_counts().head(15)
    ax.barh(range(len(vc)), vc.values)
    ax.set_yticks(range(len(vc)))
    ax.set_yticklabels([textwrap.fill(str(v), 30) for v in vc.index], fontsize=7)
    ax.set_title(f'Provider {col}')
    ax.invert_yaxis()

plt.tight_layout(); plt.show()

> **Note:** `PrimarySpecialty` is 95.2% `*Not Applicable` or `*Unknown`. Use `DepartmentSpecialty` from the departments table for specialty-level analysis.

---
## 7. Departments

In [ ]:
dept.head(3)

In [ ]:
# Department types
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vc_type = dept['DepartmentType'].value_counts().head(12)
axes[0].bar(range(len(vc_type)), vc_type.values)
axes[0].set_xticks(range(len(vc_type)))
axes[0].set_xticklabels(vc_type.index, rotation=45, ha='right', fontsize=8)
axes[0].set_title('Department Types (count of depts)')

# Encounter volume per department type
enc_dept = enc.merge(dept[['DepartmentKey', 'DepartmentType']], on='DepartmentKey', how='left')
vc_enc = enc_dept['DepartmentType'].value_counts().head(12)
axes[1].barh(range(len(vc_enc)), vc_enc.values)
axes[1].set_yticks(range(len(vc_enc)))
axes[1].set_yticklabels(vc_enc.index, fontsize=8)
axes[1].set_title('Encounter Volume by Department Type')
axes[1].invert_yaxis()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))

plt.tight_layout(); plt.show()

---
## 8. Social Determinants of Health (SDOH)

**Important prep steps:**
1. Filter out `DisplayName == '*Unspecified'` (58.5% of rows)
2. The `Domain` column is 58.5% null — identify domain from `DisplayName` instead
3. Survey rollout was gradual — hospital admissions first, then expanded

In [ ]:
# Clean SDOH — remove unspecified display names
sdoh_clean = sdoh[~sdoh['DisplayName'].astype(str).str.startswith('*')].copy()
print(f'Raw SDOH rows:    {len(sdoh):>9,}')
print(f'After filtering:  {len(sdoh_clean):>9,}  ({len(sdoh_clean)/len(sdoh)*100:.1f}% retained)')

In [ ]:
# Domain coverage (from Domain column where available)
domain_counts = sdoh_clean['Domain'].value_counts(dropna=True)
print('SDOH Domain response counts:')
print(domain_counts.to_string())

In [ ]:
# Top survey questions
q_counts = sdoh_clean['DisplayName'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 8))
q_counts.plot(kind='barh', ax=ax, color=sns.color_palette('muted')[5])
ax.set_title('Top 20 SDOH Survey Questions by Response Count')
ax.invert_yaxis()
ax.set_xlabel('Response Count')
ax.yaxis.set_tick_params(labelsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Answer distributions for the top 6 questions
top_q = q_counts.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, q in zip(axes.flatten(), top_q):
    ans = sdoh_clean[sdoh_clean['DisplayName'] == q]['AnswerText'].value_counts().head(8)
    ax.barh(range(len(ans)), ans.values)
    ax.set_yticks(range(len(ans)))
    ax.set_yticklabels([textwrap.fill(str(v), 25) for v in ans.index], fontsize=7)
    ax.set_title(textwrap.fill(q, 45), fontsize=8)
    ax.invert_yaxis()

plt.suptitle('Answer Distributions — Top 6 SDOH Questions', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# SDOH rollout over time
sdoh_enc = sdoh_clean.merge(enc[['EncounterKey', 'Date']], on='EncounterKey', how='left')
sdoh_enc['Date'] = pd.to_datetime(sdoh_enc['Date'], errors='coerce')
sdoh_enc['YearMonth'] = sdoh_enc['Date'].dt.to_period('M')

monthly = sdoh_enc.groupby('YearMonth').size().reset_index(name='responses')
monthly = monthly[monthly['YearMonth'].notna()]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly['YearMonth'].astype(str), monthly['responses'],
        marker='o', markersize=3, color=sns.color_palette('muted')[5])
ax.set_title('SDOH Survey Response Volume Over Time (Rollout Tracking)')
ax.set_xlabel('Month'); ax.set_ylabel('Survey Responses')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
n_sdoh_pts = sdoh['PatientDurableKey'].nunique()
n_all_pts  = pat['DurableKey'].nunique()
print(f'Patients with ≥1 SDOH response: {n_sdoh_pts:,} ({n_sdoh_pts/n_all_pts*100:.1f}% of all patients)')

---
## 9. Join Validation

Check referential integrity across all key joins.

In [ ]:
join_checks = [
    ('encounters.PatientDurableKey',          enc['PatientDurableKey'].nunique(),
     'patients.DurableKey',                   pat['DurableKey'].nunique(),
     enc['PatientDurableKey'].isin(pat['DurableKey']).mean() * 100),

    ('encounters.DepartmentKey',              enc['DepartmentKey'].nunique(),
     'departments.DepartmentKey',             dept['DepartmentKey'].nunique(),
     enc['DepartmentKey'].isin(dept['DepartmentKey']).mean() * 100),

    ('encounters.PrimaryDiagnosisKey (≠-1)',
     enc[enc.PrimaryDiagnosisKey != -1]['PrimaryDiagnosisKey'].nunique(),
     'diagnosis.DiagnosisKey',                diag['DiagnosisKey'].nunique(),
     enc[enc.PrimaryDiagnosisKey != -1]['PrimaryDiagnosisKey'].isin(diag['DiagnosisKey']).mean() * 100),

    ('encounters.ProviderDurableKey',         enc['ProviderDurableKey'].nunique(),
     'providers.DurableKey',                  prov['DurableKey'].nunique(),
     enc['ProviderDurableKey'].isin(prov['DurableKey']).mean() * 100),

    ('social_determinants.PatientDurableKey', sdoh['PatientDurableKey'].nunique(),
     'patients.DurableKey',                   pat['DurableKey'].nunique(),
     sdoh['PatientDurableKey'].isin(pat['DurableKey']).mean() * 100),
]

jdf = pd.DataFrame(join_checks,
    columns=['from_key', 'from_unique', 'to_key', 'to_unique', 'match_pct'])
jdf['match_pct'] = jdf['match_pct'].round(2)
jdf

---
## 10. Patient Journey Setup

A **patient journey** = all encounters for a given patient, sorted by date, linked to a diagnosis.

In [ ]:
# Encounters per patient
enc_per_patient = enc.groupby('PatientDurableKey').size()

print('Encounters per patient:')
print(enc_per_patient.describe().round(1))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(enc_per_patient.clip(upper=100), bins=50,
        color=sns.color_palette('muted')[0], edgecolor='white')
ax.set_title('Encounters per Patient (capped at 100)')
ax.set_xlabel('Encounters'); ax.set_ylabel('Patients')
plt.tight_layout(); plt.show()

In [ ]:
# Enrich encounters with diagnosis group
enc_diag = enc.merge(
    diag[['DiagnosisKey', 'GroupCode', 'GroupName', 'DiagnosisValue']],
    left_on='PrimaryDiagnosisKey', right_on='DiagnosisKey', how='left'
)

# Journey length by top 10 diagnosis groups
top_groups = enc_diag['GroupName'].value_counts().head(10).index
journey_by_dx = (
    enc_diag[enc_diag['GroupName'].isin(top_groups)]
    .groupby(['PatientDurableKey', 'GroupName'])
    .size()
    .reset_index(name='n_encounters')
)

fig, ax = plt.subplots(figsize=(13, 5))
journey_by_dx.boxplot(
    column='n_encounters', by='GroupName', ax=ax,
    flierprops=dict(marker='.', markersize=2, alpha=0.3)
)
ax.set_title('Encounters per Patient Journey — Top 10 Diagnosis Groups')
plt.suptitle('')
ax.set_xlabel('')
ax.set_ylabel('Encounters in Journey')
ax.set_ylim(0, 50)
ax.set_xticklabels(
    [textwrap.fill(str(t.get_text()), 20) for t in ax.get_xticklabels()],
    rotation=45, ha='right', fontsize=7
)
plt.tight_layout(); plt.show()

In [ ]:
# Inter-encounter time gaps
print('Computing inter-encounter gaps (sample of first 200K patients)...')
sample_pts = enc['PatientDurableKey'].unique()[:200_000]
enc_s = enc[enc['PatientDurableKey'].isin(sample_pts)].copy()
enc_s = enc_s.sort_values(['PatientDurableKey', 'Date'])
enc_s['days_since_last'] = enc_s.groupby('PatientDurableKey')['Date'].diff().dt.days
gaps = enc_s['days_since_last'].dropna()

print('\nTime gap (days) between consecutive encounters:')
print(gaps.describe().round(1))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(gaps.clip(upper=365), bins=73,
        color=sns.color_palette('muted')[2], edgecolor='white')
ax.set_title('Days Between Consecutive Encounters (capped at 365 days)')
ax.set_xlabel('Days'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

---
## 11. Demographics × Encounter Volume

In [ ]:
# Merge patient demographics into encounters
enc_pat = enc.merge(
    pat[['DurableKey', 'PatientBirthYearBin', 'FirstRace', 'SmokingStatus', 'VitalStatus']],
    left_on='PatientDurableKey', right_on='DurableKey', how='left'
)

# Encounter volume by race
race_enc = enc_pat['FirstRace'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(range(len(race_enc)), race_enc.values, color=sns.color_palette('muted')[1])
axes[0].set_yticks(range(len(race_enc)))
axes[0].set_yticklabels(race_enc.index, fontsize=8)
axes[0].set_title('Total Encounter Volume by Race')
axes[0].invert_yaxis()

# Median encounters per patient by race
enc_count_pp = enc_pat.groupby(['PatientDurableKey', 'FirstRace']).size().reset_index(name='n_enc')
race_median  = enc_count_pp.groupby('FirstRace')['n_enc'].median().sort_values(ascending=False).head(10)

axes[1].bar(range(len(race_median)), race_median.values, color=sns.color_palette('muted')[1])
axes[1].set_xticks(range(len(race_median)))
axes[1].set_xticklabels(
    [textwrap.fill(str(v), 18) for v in race_median.index],
    rotation=45, ha='right', fontsize=7
)
axes[1].set_title('Median Encounters per Patient by Race')
axes[1].set_ylabel('Median Encounters')

plt.suptitle('Race × Encounter Patterns')
plt.tight_layout(); plt.show()

print('\nMedian encounters per patient by race:')
print(race_median)

---
## 12. Your Analysis Starts Here

Use the enriched base tables below as your starting point.

**Helpful pre-built joins:**

In [ ]:
# Fully enriched encounter base — add columns as needed for your question
base = (
    enc
    .merge(pat[['DurableKey', 'PatientBirthYearBin', 'FirstRace',
                'SmokingStatus', 'VitalStatus', 'MyChartStatus']],
           left_on='PatientDurableKey', right_on='DurableKey', how='left')
    .merge(diag[['DiagnosisKey', 'GroupCode', 'GroupName', 'DiagnosisValue', 'DiagnosisName']],
           left_on='PrimaryDiagnosisKey', right_on='DiagnosisKey', how='left')
    .merge(dept[['DepartmentKey', 'DepartmentType', 'DepartmentSpecialty']],
           on='DepartmentKey', how='left')
)

print(f'Enriched base: {len(base):,} rows × {base.shape[1]} cols')
base.head(3)

In [ ]:
# Journey table — sorted encounters per patient with rank and gap
# NOTE: A single clinic visit can produce multiple encounter rows (e.g. physician referral
# to radiology on the same day = 2 rows, same PrimaryDiagnosisKey). Do not treat every
# encounter as a distinct visit when counting patient interactions.
journey = (
    base
    .sort_values(['PatientDurableKey', 'Date'])
    .assign(
        enc_rank        = lambda df: df.groupby('PatientDurableKey').cumcount() + 1,
        days_since_last = lambda df: df.groupby('PatientDurableKey')['Date'].diff().dt.days
    )
)

print(f'Journey table: {len(journey):,} rows')
journey[['PatientDurableKey', 'Date', 'enc_rank', 'days_since_last',
          'Type', 'GroupName']].head(10)

In [ ]:
# SDOH-linked encounters (for patients with social determinant data)
sdoh_base = (
    sdoh_clean
    .merge(enc[['EncounterKey', 'Date', 'PatientDurableKey', 'Type']],
           on='EncounterKey', how='left')
    .merge(pat[['DurableKey', 'FirstRace', 'PatientBirthYearBin']],
           left_on='PatientDurableKey', right_on='DurableKey', how='left')
)

print(f'SDOH-linked table: {len(sdoh_base):,} rows')
sdoh_base.head(3)

---
## Suggested Next Steps

Pick a research angle and build from `base`, `journey`, or `sdoh_base`:

**Patient Journey Analysis**
- Compare journey length (# encounters, total days) across diagnosis groups
- Identify patients with unusually long gaps — what predicts delayed follow-up?
- Find "redundant" encounter patterns (same type repeated with no progress)

**Health Equity**
- Do racial/ethnic groups differ in journey length or LOS?
- Do transportation-barrier patients miss more follow-up appointments?
- Rural vs. urban geography (via `CensusBlockGroupFipsCode` → `tigercensuscodes`)

**SDOH Impact**
- Correlate food insecurity / housing instability with hospital readmissions
- Does MyChart activation predict better follow-up adherence?

**Operational Patterns**
- Seasonal / day-of-week patterns in ED visits
- Which conditions show the longest diagnostic delays?

---
*ASA DataFest 2026 — Stormont Vail Health*  
*Data is proprietary. Do not upload to public services or external APIs.*